# Notebook 3 — Model Training

This notebook trains two deep-learning architectures on the FER2013 dataset
and saves the resulting models and training histories to disk.

- **Model A — Custom CNN**: a four-block convolutional network trained from
  scratch on 48 × 48 grayscale images. Serves as the interpretable baseline.
- **Model B — MobileNetV2 fine-tuned**: a lightweight pretrained backbone
  adapted to FER2013 in two stages. Expected to generalise better due to
  ImageNet pretraining.

All evaluation (accuracy curves, confusion matrices, per-class metrics) is
deferred to Notebook 4. Here we only run a brief smoke test after each model
is saved to confirm the output pipeline works end-to-end.

## Section 1 — Setup

We fix all random seeds before any other operation to ensure reproducibility
across runs. Seeds are set for Python's `random` module, NumPy, and TensorFlow
independently because each maintains its own PRNG state. We also verify GPU
availability here; training without a GPU is possible but wall-clock time
increases roughly 10×.

In [1]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.models.cnn_custom import build_cnn_custom
from src.models.mobilenet_finetune import build_mobilenet_head, unfreeze_top_layers
from src.data.augmentation import build_augmentation_pipeline
from src.models.trainer import get_default_callbacks, save_history, train_model

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {gpus}")
else:
    print("WARNING: No GPU detected. Training will be significantly slower.")

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Section 2 — Load preprocessed data

We reload the `.npz` files and class-weights JSON produced by Notebook 2
rather than re-running the preprocessing pipeline. This ensures that the
exact same normalised tensors and train/val/test split are used for both
models, and reduces notebook startup time from several minutes to a few
seconds. JSON dictionary keys are strings by default; we cast them to `int`
because the Keras `class_weight` argument requires integer keys.

In [ ]:
gray = np.load(config.PROCESSED_DIR / "fer2013_gray.npz")
rgb  = np.load(config.PROCESSED_DIR / "fer2013_rgb96.npz")

with open(config.PROCESSED_DIR / "class_weights.json") as f:
    class_weights_raw = json.load(f)
class_weights = {int(k): v for k, v in class_weights_raw.items()}

X_train_gray, y_train = gray["X_train"], gray["y_train"]
X_val_gray,   y_val   = gray["X_val"],   gray["y_val"]
X_test_gray,  y_test  = gray["X_test"],  gray["y_test"]

X_train_rgb = rgb["X_train"]
X_val_rgb   = rgb["X_val"]
X_test_rgb  = rgb["X_test"]

print(f"Grayscale  — train: {X_train_gray.shape}  val: {X_val_gray.shape}  test: {X_test_gray.shape}")
print(f"RGB 96×96  — train: {X_train_rgb.shape}  val: {X_val_rgb.shape}  test: {X_test_rgb.shape}")
print(f"Labels     — y_train: {y_train.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}")
print(f"Class weights: {class_weights}")

In [3]:
# Cap class weights: raw balanced weights give Disgust 9.4x which destabilises training.
# Capping at 3x keeps the minority-class signal without overwhelming the optimiser.
class_weights = {k: min(v, 3.0) for k, v in class_weights.items()}
print("Capped class weights:")
for idx, label in enumerate(config.EMOTION_LABELS):
    print(f"  {idx}  {label:<10}  {class_weights[idx]:.4f}")

Capped class weights:
  0  angry       1.0265
  1  disgust     3.0000
  2  fear        1.0011
  3  happy       0.5684
  4  sad         0.8492
  5  surprise    1.2935
  6  neutral     0.8261


FER2013 is class-imbalanced (*Happy* ~25 %, *Disgust* ~1.5 %). The
intuitive fix — pass `class_weight` or `sample_weight` to `model.fit()` —
does not work on TF 2.10 with one-hot labels: both routes hit a known
weighting bug where the optimiser falls into a degenerate uniform-output
minimum in epoch 1 and never escapes. We therefore train **without
weighting** and let augmentation plus a sufficiently long patience
absorb the imbalance. The cell below still computes the weights so the
class-frequency information is available, but they are not passed to
`fit()`. Expect a 6–8 epoch plateau at *Happy*-only predictions before
the optimiser escapes and starts learning the minority classes.

In [4]:
y_train_idx = np.argmax(y_train, axis=1)
sample_weight_train = np.array([class_weights[c] for c in y_train_idx], dtype=np.float32)
print(f"sample_weight_train — shape: {sample_weight_train.shape}")
print(f"  min={sample_weight_train.min():.4f}  max={sample_weight_train.max():.4f}  mean={sample_weight_train.mean():.4f}")
print("NOTE: not passed to fit() — TF 2.10 weighting bug. Kept for reference only.")

sample_weight_train — shape: (24402,)
  min=0.5684  max=3.0000  mean=0.9028
NOTE: not passed to fit() — TF 2.10 weighting bug. Kept for reference only.


## Section 3 — Custom CNN (Model A)

### Section 3.1 — Build the architecture

The custom CNN uses four convolutional blocks with doubling filter counts
(32 → 64 → 128 → 256), each followed by BatchNorm, ReLU, MaxPooling, and
**Dropout(0.1)**. Two dense layers (512, 256) with **Dropout(0.3)** precede
the 7-class softmax head. Loss is **categorical cross-entropy with label
smoothing 0.1** — FER2013 contains genuinely ambiguous expressions
(Sad/Neutral, Angry/Disgust) and a small smoothing term prevents the model
from collapsing to one-hot predictions on those borderline samples,
typically gaining 0.5–1 pt of val_accuracy. Dropout is kept light: with
the augmentation pipeline (Section 3.3) already injecting noise, the
more conventional 0.25 / 0.5 dropout values pile so much regularisation
onto the early gradients that the optimiser never escapes the majority-class
(*Happy*) minimum.

In [5]:
model_cnn = build_cnn_custom(input_shape=(48, 48, 1), num_classes=7)
model_cnn.summary()

Model: "cnn_custom"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 48, 48, 1)]       0         
                                                                 
 conv2d (Conv2D)             (None, 48, 48, 32)        320       
                                                                 
 batch_normalization (BatchN  (None, 48, 48, 32)       128       
 ormalization)                                                   
                                                                 
 re_lu (ReLU)                (None, 48, 48, 32)        0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 24, 24, 32)       0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 24, 24, 32)        0

### Section 3.2 — Configure training callbacks

Three standard callbacks are attached. `EarlyStopping` (patience=12) halts
training when validation loss stagnates and restores the best weights seen
during the run. `ReduceLROnPlateau` (patience=5, factor=0.5) halves the
learning rate after five epochs without improvement, letting the optimiser
converge into narrow loss minima that a large fixed LR would overshoot.
Patience is set wide on both because the model spends 6–8 initial epochs
in a *Happy*-only plateau (val_loss flat); shorter patience would kill
training or drop the LR before the optimiser escapes.
`ModelCheckpoint` writes the best checkpoint independently, so it is
recoverable even if the process is interrupted.

In [6]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

callbacks_cnn = get_default_callbacks("cnn_custom", config.MODELS_DIR)
print("Callbacks configured:")
for cb in callbacks_cnn:
    print(f"  {cb.__class__.__name__}")

Callbacks configured:
  EarlyStopping
  ReduceLROnPlateau
  ModelCheckpoint


### Section 3.3 — Train the custom CNN

Up to 75 epochs with batch size 64 — extended from the original 50 because
the model was still climbing at epoch 50 (final val_loss still improving),
so EarlyStopping is the natural cap rather than the hard epoch limit.
Images are augmented on-the-fly using the cv2-based `AugmentationPipeline`.
No `sample_weight` or `class_weight` is passed: on TF 2.10 with one-hot
labels, both routes hit a known weighting bug that collapses training to
uniform output. Class imbalance is instead absorbed by augmentation + the
lighter dropout configured in Section 3.1. The model briefly plateaus on
*Happy*-only predictions (val_accuracy ≈ 0.25 ≈ Happy's frequency) for
2–3 epochs, then escapes and climbs steadily. Expected validation
accuracy at convergence: **60–65 %**.

In [ ]:
datagen_gray = build_augmentation_pipeline()
train_gen_cnn = datagen_gray.flow(X_train_gray, y_train, batch_size=config.BATCH_SIZE, seed=SEED)

history_cnn = train_model(
    model=model_cnn,
    train_data=train_gen_cnn,
    val_data=(X_val_gray, y_val),
    class_weights=None,  # TF 2.10: any weighting + one-hot labels causes degenerate training
    epochs=75,
    callbacks=callbacks_cnn,
    batch_size=config.BATCH_SIZE,
)
print(f"Training complete — {len(history_cnn.history['loss'])} epochs run.")

### Section 3.4 — Save the custom CNN

Two artefacts are persisted: the final model (used by the inference pipeline
and the API service) and the training history as JSON (used by Notebook 4 to
plot learning curves without retraining). The `.keras` format is preferred
on TF 2.10; if saving fails the cell falls back to `.h5` and logs the reason.
Both model and history paths are printed so they can be verified on disk.

In [8]:
model_path_cnn = config.MODELS_DIR / "cnn_custom.keras"
try:
    model_cnn.save(str(model_path_cnn))
    print(f"Saved model: {model_path_cnn}")
except Exception as e:
    model_path_cnn = config.MODELS_DIR / "cnn_custom.h5"
    model_cnn.save(str(model_path_cnn))
    print(f"Fell back to .h5 — saved: {model_path_cnn}  (reason: {e})")

history_path_cnn = config.HISTORIES_DIR / "cnn_custom_history.json"
save_history(history_cnn, history_path_cnn)
print(f"Saved history ({len(history_cnn.history['loss'])} epochs): {history_path_cnn}")

Saved model: C:\Users\jpulg\Documents\Todo\Proyectos\sales-receptivity-cnn\models\cnn_custom.keras
Saved history (50 epochs): C:\Users\jpulg\Documents\Todo\Proyectos\sales-receptivity-cnn\models\histories\cnn_custom_history.json


## Section 4 — MobileNetV2 Fine-tuned (Model B)

### Section 4.1 — Build the MobileNetV2 head

A small classification head is attached to a frozen MobileNetV2 backbone
loaded with ImageNet weights. Input resolution is **96 × 96 × 3** — the
official minimum useful size for MobileNetV2. The earlier attempt at 64×64
left the backbone with only a 2×2 spatial feature map after its ~32×
downsampling, which is barely two distinct regions for `GlobalAveragePooling2D`
to average; emotion classification stalled at 38 % val_acc (Stage 1) /
45 % (Stage 2). At 96×96 the backbone outputs a 3×3 feature map and the
head has meaningfully more spatial structure to learn from.

At this stage only the `GlobalAveragePooling2D → Dense(256) → Dropout(0.3)
→ Dense(7)` head is trainable; the backbone's 154 layers are frozen.
Dropout is **0.3** (lighter than the original 0.5) to match the same
augmentation-vs-regularisation balance that fixed the custom CNN.
Label smoothing 0.1 in the loss for the same reason as the CNN.

In [ ]:
model_mob = build_mobilenet_head(input_shape=(96, 96, 3), num_classes=7)
model_mob.summary()

trainable_params = np.sum([np.prod(v.shape) for v in model_mob.trainable_weights])
frozen_params    = np.sum([np.prod(v.shape) for v in model_mob.non_trainable_weights])
print(f"\nTrainable params:  {trainable_params:,}")
print(f"Frozen params:     {frozen_params:,}")

### Section 4.2 — Stage 1: train the classification head

The head is trained for **20 epochs** at lr = 1e-3 before any backbone layers
are touched (extended from the original 10 — the previous run was still
improving when it stopped). Images are augmented on-the-fly with no
weighting (same rationale as the custom CNN: TF 2.10 sample/class
weighting + one-hot labels collapses training). This warm-up lets the
randomly-initialised dense layers stabilise their weights before
fine-tuning begins; without it, large head gradients would corrupt the
pretrained backbone features during the first few backward passes.

In [ ]:
callbacks_mob = get_default_callbacks("mobilenet_ft", config.MODELS_DIR)

datagen_rgb = build_augmentation_pipeline()
train_gen_mob = datagen_rgb.flow(X_train_rgb, y_train, batch_size=config.BATCH_SIZE, seed=SEED)

history_mob_s1 = train_model(
    model=model_mob,
    train_data=train_gen_mob,
    val_data=(X_val_rgb, y_val),
    class_weights=None,  # TF 2.10: any weighting + one-hot labels causes degenerate training
    epochs=20,
    callbacks=callbacks_mob,
    batch_size=config.BATCH_SIZE,
)
print(f"\nStage 1 done — final val_accuracy: {history_mob_s1.history['val_accuracy'][-1]:.4f}")

### Section 4.3 — Stage 2: unfreeze top layers and fine-tune

The top **60** backbone layers are unfrozen (~40 % of MobileNetV2's 154
layers, up from the original 30) and the model recompiled with
**lr = 5e-5**. Reasoning: FER2013 faces look nothing like ImageNet objects,
so the highest-level filters are the ones most worth re-tuning. The
slightly higher LR (5e-5 vs the conventional 1e-5) is safe because Stage 1
already produced a well-conditioned head — small gradients would barely
move the backbone weights. The same augmentation generator from Stage 1
is reused, so the backbone sees varied images throughout fine-tuning.
The parameter count printed below confirms that trainable params increase
substantially after the unfreeze.

In [ ]:
model_mob = unfreeze_top_layers(model_mob, n_layers=60)

trainable_ft = np.sum([np.prod(v.shape) for v in model_mob.trainable_weights])
frozen_ft    = np.sum([np.prod(v.shape) for v in model_mob.non_trainable_weights])
print(f"Trainable params after unfreeze: {trainable_ft:,}")
print(f"Frozen params after unfreeze:    {frozen_ft:,}")

history_mob_s2 = train_model(
    model=model_mob,
    train_data=train_gen_mob,
    val_data=(X_val_rgb, y_val),
    class_weights=None,  # TF 2.10: any weighting + one-hot labels causes degenerate training
    epochs=30,
    callbacks=callbacks_mob,
    batch_size=config.BATCH_SIZE,
)
print(f"\nStage 2 done — final val_accuracy: {history_mob_s2.history['val_accuracy'][-1]:.4f}")

### Section 4.4 — Save the fine-tuned MobileNetV2

The Stage 1 and Stage 2 histories are concatenated before saving: for each
metric key, the two value lists are appended to form a single sequence
covering all training epochs. This gives Notebook 4 one continuous learning
curve per metric without needing to know about the two-stage protocol.
The total epoch count printed below should equal Stage 1 + Stage 2 epochs
that actually ran (early stopping may reduce the Stage 2 count).

In [12]:
model_path_mob = config.MODELS_DIR / "mobilenet_ft.keras"
try:
    model_mob.save(str(model_path_mob))
    print(f"Saved model: {model_path_mob}")
except Exception as e:
    model_path_mob = config.MODELS_DIR / "mobilenet_ft.h5"
    model_mob.save(str(model_path_mob))
    print(f"Fell back to .h5 — saved: {model_path_mob}  (reason: {e})")

combined_history: dict = {
    key: history_mob_s1.history[key] + history_mob_s2.history[key]
    for key in history_mob_s1.history
}
history_path_mob = config.HISTORIES_DIR / "mobilenet_ft_history.json"
serializable = {k: [float(v) for v in vals] for k, vals in combined_history.items()}
with open(history_path_mob, "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2)
total_epochs = len(combined_history["loss"])
print(f"Saved combined history ({total_epochs} epochs total): {history_path_mob}")

Saved model: C:\Users\jpulg\Documents\Todo\Proyectos\sales-receptivity-cnn\models\mobilenet_ft.keras
Saved combined history (30 epochs total): C:\Users\jpulg\Documents\Todo\Proyectos\sales-receptivity-cnn\models\histories\mobilenet_ft_history.json


## Section 5 — Quick sanity check

Both models predict the emotion label for the first 16 test images. The goal
is not to measure accuracy (that is the subject of Notebook 4) but to confirm
that each model produces a valid 7-class output and that the argmax pipeline
works correctly. A model that predicts the same class for every input would
signal a training failure worth investigating before proceeding.

In [13]:
LABELS = config.EMOTION_LABELS

def _smoke_test(model, X_subset, y_subset, name):
    preds = model.predict(X_subset, verbose=0)
    pred_cls = np.argmax(preds, axis=1)
    true_cls = np.argmax(y_subset, axis=1)
    print(f"\n{name} — predictions on first 16 test samples:")
    print(f"  {'#':>2}  {'Predicted':<12} {'True':<12} Match")
    print("  " + "-" * 38)
    for i, (p, t) in enumerate(zip(pred_cls, true_cls)):
        mark = "v" if p == t else "x"
        print(f"  {i:>2}  {LABELS[p]:<12} {LABELS[t]:<12} {mark}")

_smoke_test(model_cnn, X_test_gray[:16], y_test[:16], "Custom CNN")
_smoke_test(model_mob, X_test_rgb[:16],  y_test[:16], "MobileNetV2")


Custom CNN — predictions on first 16 test samples:
   #  Predicted    True         Match
  --------------------------------------
   0  angry        angry        v
   1  angry        angry        v
   2  sad          angry        x
   3  sad          angry        x
   4  angry        angry        v
   5  sad          angry        x
   6  neutral      angry        x
   7  angry        angry        v
   8  sad          angry        x
   9  angry        angry        v
  10  fear         angry        x
  11  sad          angry        x
  12  sad          angry        x
  13  angry        angry        v
  14  fear         angry        x
  15  angry        angry        v

MobileNetV2 — predictions on first 16 test samples:
   #  Predicted    True         Match
  --------------------------------------
   0  angry        angry        v
   1  angry        angry        v
   2  sad          angry        x
   3  sad          angry        x
   4  sad          angry        x
   5  angry        angr

## Section 6 — Summary and link to the next notebook

This notebook trained and saved both FER2013 classifiers:

| Artefact | Path |
|----------|------|
| Custom CNN model |  (or  fallback) |
| Custom CNN history |  |
| MobileNetV2 model |  (or  fallback) |
| MobileNetV2 history |  |

Both models were trained **without** class or sample weighting (TF 2.10 +
one-hot labels collapses weighted training into a uniform-output minimum),
with **light dropout** (0.1 / 0.3 for the CNN; 0.3 in the MobileNet head)
to keep total regularisation compatible with the augmentation noise, and
with **label smoothing 0.1** in both losses to soften over-confident
predictions on ambiguous expressions.

The MobileNetV2 input was raised from 64×64 to **96×96** to give its
pretrained backbone a non-degenerate spatial feature map to work with;
Stage 1 was extended to 20 epochs, Stage 2 to 30 epochs with 60 unfrozen
backbone layers (~40 %) at lr = 5e-5. On a modern NVIDIA GPU (RTX 3060
or better) expect roughly 5–10 min for the custom CNN and 15–25 min for
MobileNetV2 (both stages combined).

**Next: Notebook 4 — Evaluation** ().
It loads the saved models and history files, produces accuracy/loss curves,
confusion matrices, per-class precision/recall/F1, and the final
receptivity-index visualisation linking model outputs to the sales domain.